# Instruction-Tuned ML & Data Science Assistant

This notebook is the clean, modular replacement for the original InstructionTune 360 prototype. It uses the project source modules instead of duplicating logic.

> **Responsible use:** Educational portfolio demonstration only. Do not use generated responses for legal, medical, financial, immigration, safety-critical, or official decisions. Do not paste private or confidential data.


## 1. Project imports and configuration

In [ ]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PROJECT_ROOT

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import ModelConfig, LoraTrainingConfig
from src.data_preprocessing import load_jsonl, validate_and_clean_records, split_records
from src.prompt_templates import build_training_prompt
from src.visualization import create_dataset_visualizations

## 2. Load and validate the custom ML/DS instruction dataset

In [ ]:
records = load_jsonl(PROJECT_ROOT / 'data' / 'ml_ds_instruction_dataset.jsonl')
cleaned, report = validate_and_clean_records(records)
report.to_dict()

In [ ]:
df = pd.DataFrame(cleaned)
print(df.shape)
display(df.head())
display(df['category'].value_counts())

In [ ]:
splits = split_records(cleaned)
{k: len(v) for k, v in splits.items()}

## 3. Verify the shared training and inference prompt template

In [ ]:
example = cleaned[0]
print(build_training_prompt(example['instruction'], example['input'], example['category']))
print('\nTARGET:\n', example['output'])

## 4. Create dataset visualizations

In [ ]:
create_dataset_visualizations(cleaned, PROJECT_ROOT / 'outputs')
print('Saved visualizations to outputs/')

## 5. Review LoRA settings

In [ ]:
ModelConfig(), LoraTrainingConfig()

## 6. Train the adapter on a GPU

Training is intentionally not executed automatically in this notebook. Run the command below in Colab, Kaggle, or a local GPU environment:

```bash
python scripts/train_lora.py --output-dir models/training_run
```

The command saves the LoRA adapter, tokenizer, trainer state, and model metadata. The Hugging Face Space performs inference only.


In [ ]:
# Optional GPU training call:
# from src.model_training import train_lora_adapter
# metadata = train_lora_adapter(
#     PROJECT_ROOT / 'data' / 'ml_ds_instruction_dataset.jsonl',
#     PROJECT_ROOT / 'models' / 'training_run',
# )
# metadata

## 7. Configure adapter-backed inference

In [ ]:
# Set before loading the assistant, for example:
# import os
# os.environ['ADAPTER_ID'] = '<username>/ml-ds-flan-t5-lora'
# from src.inference_pipeline import InstructionAssistant
# assistant = InstructionAssistant()
# assistant.generate('Explain precision and recall.', category='Metric explanation')

## 8. Next step
Run `scripts/evaluate_model.py --bertscore`, review the manual evaluation CSV, and deploy `app.py` to Hugging Face Spaces.